In [1]:
import pandas as pd

In [4]:
user_counts = {}

for chunk in pd.read_csv("User Listening History.csv", chunksize=1000000):
    counts = chunk["user_id"].value_counts()
    for user_id, count in counts.items():
        user_counts[user_id] = user_counts.get(user_id, 0) + count
len(user_counts)

valid_users = {user for user, count in user_counts.items() if count >= 10}
print(len(valid_users))

290898


In [ ]:
first_chunk = True

for chunk in pd.read_csv("User Listening History.csv", chunksize=1000000):
    filtered = chunk[chunk["user_id"].isin(valid_users)]
    filtered.to_csv(
        "listening_history_clean.csv", mode="a", header=first_chunk, index=False
    )
    first_chunk = False


In [6]:
result = pd.read_csv("listening_history_clean.csv")
print(result.shape)
print(result["user_id"].nunique())


(7070360, 3)
290898


In [9]:
user_ids = result["user_id"].unique()
track_ids = result["track_id"].unique()

user_to_idx = {user: idx for idx, user in enumerate(user_ids)}
track_to_idx = {track: idx for idx, track in enumerate(track_ids)}

row_indices = result["user_id"].map(user_to_idx)
col_indices = result["track_id"].map(track_to_idx)
values = result["playcount"]

from scipy.sparse import csr_matrix

interaction_matrix = csr_matrix(
    (values, (row_indices, col_indices)), shape=(len(user_ids), len(track_ids))
)

print(interaction_matrix.shape)
print(interaction_matrix.nnz)
print(interaction_matrix.data.nbytes / 1e6, "MB")  # actual stored data

(290898, 29922)
7070360
56.56288 MB
